# Comparing PhaseNet weight sets on Ridgecrest

Runs two or more PhaseNet weight sets over identical waveforms and
compares what they pick.

The M4.6 aftershock is used rather than the mainshock: its short,
impulsive source produces clean P **and** S at these distances, which
discriminates between weight sets far better than a magnitude 7 whose
S arrival is buried in rupture radiation.

| Weight set | What it is |
|---|---|
| `original` | Published Zhu & Beroza (2019) PhaseNet, trained on Southern California |
| `scedc` | Trained on the SCEDC archive — same network and region as this data |
| `instance` | Trained on the Italian INSTANCE dataset — an out-of-region control |
| `jma_wc` | Japanese model, the parent of the QuakeScope 2026 fine-tune (see note) |
| `quakescope2026` | The fine-tuned weights, included automatically if installed locally |

`original` and `scedc` both have the home-field advantage on Southern
California data while `instance` does not, which makes the three together
a rough read on how much regional training matters here.

> **Note on `jma_wc`.** Some SeisBench releases ship this weight with a
> version string the packaging library cannot parse, and loading it
> raises `InvalidVersion: '1.partial'`. The loader below skips any weight
> that fails rather than aborting the notebook, so this degrades to a
> message instead of a crash.

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import seisbench.models as sbm
from obspy import UTCDateTime
from obspy.geodetics import gps2dist_azimuth
from s3fs import S3FileSystem

%matplotlib inline

## 1. Configuration

In [ ]:
# --- Events (USGS catalog) --------------------------------------------------
# Both fall on 2019 day-of-year 187, so they share the same SCEDC day files.
MAINSHOCK = dict(
    name="M7.1 mainshock",
    time=UTCDateTime("2019-07-06T03:19:53.040000Z"),
    lat=35.770, lon=-117.599, depth_km=8.0,
)
AFTERSHOCK = dict(
    name="M4.6 aftershock",
    time=UTCDateTime("2019-07-06T08:32:57.550000Z"),
    lat=35.639, lon=-117.491, depth_km=3.1,
)

YEAR, DOY = 2019, 187

# --- Stations: CI (SCSN), all verified present in SCEDC on this day --------
STATIONS = [
    ("CLC",  35.8157, -117.5975),
    ("TOW2", 35.8086, -117.7649),
    ("SRT",  35.6923, -117.7505),
    ("WRC2", 35.9479, -117.6504),
    ("JRC2", 35.9825, -117.8089),
]

NETWORK, CHANNEL = "CI", "HH"      # broadband, 100 Hz
P_THRESHOLD = S_THRESHOLD = 0.3
VP, VS = 6.0, 3.5                  # crustal averages for reference curves

# Categorical colors, CVD-validated. Phase is also encoded by linestyle and
# text label, so identity never depends on color alone.
C_P, C_S = "#2a78d6", "#eb6834"


def distance_km(event, lat, lon):
    return gps2dist_azimuth(event["lat"], event["lon"], lat, lon)[0] / 1000.0


## 2. Data access

In [ ]:
def scedc_key(sta, comp, net=NETWORK, cha=CHANNEL, year=YEAR, doy=DOY, loc=""):
    """Build an SCEDC S3 object key.

    The SCEDC layout differs from NCEDC - the network is not a directory, the
    day folder uses an underscore, and the station is padded to five characters:

        scedc-pds/continuous_waveforms/<year>/<year>_<doy>/
        <net><sta:_<5><cha><comp><loc:_<3><year><doy>.ms

    e.g. CICLC__HHZ___2019187.ms
    """
    base = f"{net}{sta.ljust(5, '_')}{cha}{comp}{loc.ljust(3, '_')}{year}{doy:03d}.ms"
    return f"scedc-pds/continuous_waveforms/{year}/{year}_{doy:03d}/{base}"


def load_event(fs, event, pre=30, post=120):
    """Fetch every station for one event, trimmed to its time window."""
    out = {}
    for sta, lat, lon in STATIONS:
        st = obspy.Stream()
        for comp in "ZNE":
            try:
                with fs.open(scedc_key(sta, comp)) as fh:
                    st += obspy.read(io.BytesIO(fh.read()))
            except FileNotFoundError:
                print(f"  {sta} {comp}: not found")
            except Exception as exc:
                print(f"  {sta} {comp}: {type(exc).__name__}")
        if len(st) == 0:
            print(f"  {sta}: no data, skipping")
            continue
        st.merge(fill_value=0)                       # close gaps into one trace
        st.trim(event["time"] - pre, event["time"] + post)
        if len(st):
            out[sta] = st
    return out


def pick_all(model, streams):
    picks = {}
    for sta, st in streams.items():
        out = model.classify(st, P_threshold=P_THRESHOLD, S_threshold=S_THRESHOLD)
        picks[sta] = list(out.picks)
    return picks


## 3. Load the weight sets

Anything listed here that is not installed is skipped with a message
rather than silently substituted.

In [ ]:
WANTED = ["original", "scedc", "instance", "jma_wc", "quakescope2026"]

available = sbm.PhaseNet.list_pretrained()
models = {}
for name in WANTED:
    if name not in available:
        print(f"{name:<16} not in this SeisBench release - skipping")
        continue
    try:
        models[name] = sbm.PhaseNet.from_pretrained(name)
        print(f"{name:<16} loaded")
    except Exception as exc:
        print(f"{name:<16} could not load ({type(exc).__name__}) - skipping")

if not models:
    raise RuntimeError("No weight sets loaded - cannot compare")
print("\nComparing " + str(len(models)) + ": " + ", ".join(models))

## 4. Fetch once, pick with each

In [ ]:
fs = S3FileSystem(anon=True)
print(f"Fetching {AFTERSHOCK['name']} ...")
streams = load_event(fs, AFTERSHOCK, pre=30, post=90)
print(f"{len(streams)}/{len(STATIONS)} stations loaded\n")

results = {}
for name, model in models.items():
    results[name] = pick_all(model, streams)
    total = sum(len(v) for v in results[name].values())
    print(f"{name:<16} {total} picks")

## 5. Summary

In [ ]:
rows = []
for name, per_station in results.items():
    allp = [p for v in per_station.values() for p in v]
    ps = [p for p in allp if p.phase == 'P']
    ss = [p for p in allp if p.phase == 'S']
    rows.append(dict(
        weights=name, total=len(allp), P=len(ps), S=len(ss),
        mean_P_conf=round(float(np.mean([p.peak_value for p in ps])), 3) if ps else np.nan,
        mean_S_conf=round(float(np.mean([p.peak_value for p in ss])), 3) if ss else np.nan,
    ))
print(pd.DataFrame(rows).to_string(index=False))

### Arrival times per station

Agreement on the arrival time of a shared event matters more than raw
pick counts — weight sets differ mostly in how many marginal detections
they emit. Values are seconds after origin time; blank means the phase
was not picked.

In [ ]:
def first_arrivals(sta_picks, event, dist, max_ratio=2.5, pad=3.0):
    """First P after origin, and the first S that could belong to the same event.

    An S is only accepted if it lands within a physically plausible interval of
    the P. Without that guard the "next S after the P" can easily belong to a
    later aftershock, which is a real hazard in this sequence.
    """
    hyp = np.hypot(dist, event["depth_km"])
    pred_sp = hyp / VS - hyp / VP

    ps = [p for p in sta_picks if p.phase == "P" and p.peak_time >= event["time"]]
    if not ps:
        return None, None, pred_sp
    p0 = min(ps, key=lambda p: p.peak_time)

    limit = pred_sp * max_ratio + pad
    ss = [s for s in sta_picks
          if s.phase == "S"
          and 0 < (s.peak_time - p0.peak_time) <= limit]
    s0 = min(ss, key=lambda s: s.peak_time) if ss else None
    return p0, s0, pred_sp


In [ ]:
rows = []
for sta, lat, lon in STATIONS:
    if sta not in streams:
        continue
    dist = distance_km(AFTERSHOCK, lat, lon)
    row = {'station': sta, 'dist_km': round(dist, 1)}
    for name, per_station in results.items():
        p0, s0, _ = first_arrivals(per_station[sta], AFTERSHOCK, dist)
        row[f'{name}:P'] = round(p0.peak_time - AFTERSHOCK['time'], 2) if p0 else np.nan
        row[f'{name}:S'] = round(s0.peak_time - AFTERSHOCK['time'], 2) if s0 else np.nan
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

## 6. Side by side on the waveforms

One row per weight set over the same trace, so differences in pick
placement are directly visible.

In [ ]:
window = (-5, 30)

for sta, st in streams.items():
    tr = st.select(component='Z')
    if not tr:
        continue
    tr = tr[0]
    t = tr.times(reftime=AFTERSHOCK['time'])
    lat, lon = next((la, lo) for c, la, lo in STATIONS if c == sta)
    dist = distance_km(AFTERSHOCK, lat, lon)

    n = len(results)
    fig, axes = plt.subplots(n, 1, figsize=(12, 1.9 * n + 0.9), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, (name, per_station) in zip(axes, results.items()):
        ax.plot(t, tr.data, color='#3d3d3d', lw=0.6)
        for p in per_station[sta]:
            dt = p.peak_time - AFTERSHOCK['time']
            if not (window[0] <= dt <= window[1]):
                continue
            is_p = p.phase == 'P'
            ax.axvline(dt, color=C_P if is_p else C_S, lw=1.6,
                       ls='-' if is_p else '--', alpha=0.9)
        ax.set_ylabel(name, fontsize=9)
        ax.grid(alpha=0.25, lw=0.5)
        ax.tick_params(labelsize=9)

    handles = [plt.Line2D([], [], color=C_P, lw=1.6, ls='-', label='P'),
               plt.Line2D([], [], color=C_S, lw=1.6, ls='--', label='S')]
    axes[0].legend(handles=handles, loc='upper right', fontsize=9,
                   frameon=False, ncol=2)
    axes[0].set_title(f"{NETWORK}.{sta}  -  {dist:.1f} km  -  {CHANNEL}Z",
                      fontsize=11, loc='left')
    axes[-1].set_xlim(*window)
    axes[-1].set_xlabel('Seconds after origin time')
    fig.tight_layout()
    plt.show()

## Reading the comparison

Pick counts alone do not rank weight sets. A model that emits more picks
may be recovering real aftershocks or may be firing on noise, and only
the waveform panels separate those cases.

The comparisons that carry information are whether every weight set finds
the same arrival to within a few tenths of a second, whether any misses a
station its peers handle, and whether the surplus picks one model
produces land on visible energy.

A single event cannot rank weight sets. For that, use the benchmark
described in `docs/phasenet_v7_model_description.md`; this notebook
exists to catch gross breakage before a production run.